# Natural Questions Contrastive Training (Regular or Logprob-Recon)

This notebook supports two training modes on Natural Questions activations:

1. `regular`: supervised contrastive training with `ProgressiveCompressor`
2. `logprob`: contrastive training plus response-logprob reconstruction with
   `LogprobReconProgressiveCompressor`

Set `training_mode` in the config cell to switch modes.

Note: `ActivationParser` expects per-sample arrays `abstantion` and `halu_test_res`
in the eval JSON. This notebook includes a compatibility step to generate them when needed.

In [ ]:
import os
repo_root = "/mnt/home/hyang1/LLM_research/HalluLens"
os.chdir(repo_root)
print(f"cwd: {os.getcwd()}")

In [ ]:
import os
import sys
import glob
import re
import json
from pathlib import Path

import torch
import pandas as pd
from torch.utils.data import DataLoader
from loguru import logger
from tqdm.auto import tqdm

from activation_logging.activation_parser import ActivationParser
from activation_research.model import (
    ProgressiveCompressor,
    LogprobReconProgressiveCompressor,
)
from activation_research.training import (
    train_contrastive,
    train_contrastive_logprob_recon,
    _contrastive_collate_with_logprob,
)
from activation_research.metric_evaluator import MultiMetricHallucinationEvaluator

logger.remove()
logger.add(sys.stdout, level="INFO")

In [ ]:
# ---- Paths from your Natural Questions run ----
model_name = "Llama-3.1-8B-Instruct"
base_dir = Path("shared/natural_questions_logprob")
task_dir = base_dir / "natural_questions" / model_name

inference_json = str(task_dir / "generation.jsonl")
eval_json = str(task_dir / "eval_results.json")
raw_eval_jsonl = str(task_dir / "raw_eval_res.jsonl")
activations_path = str(base_dir / "activations.zarr")

# ActivationParser-compatible eval JSON will be written here.
eval_json_for_training = str(task_dir / "eval_results_for_training.json")

# ---- Training mode ----
# Options: "regular" or "logprob" ("lobprob" alias is accepted)
training_mode = "logprob"
if training_mode == "lobprob":
    training_mode = "logprob"
if training_mode not in {"regular", "logprob"}:
    raise ValueError("training_mode must be one of: 'regular', 'logprob'")
use_logprob_features = training_mode == "logprob"

# ---- Dataset parameters ----
backend = "zarr"
relevant_layers = list(range(14, 30))
target_layers = [22, 26]
num_views = 4
outlier_class = 1
fixed_layer = None
response_logprobs_top_k = 20

# ---- Preload option ----
# If True, all activations are bulk-read from zarr into RAM at dataset construction
# time.  __getitem__ becomes a pure in-memory op with no disk I/O during training,
# which eliminates the zarr per-sample seek overhead that dominates dataloader time.
#
# Requirements:
#   - Available RAM > dataset size (use check_ram=True to get an upfront estimate
#     and a MemoryError before allocation if headroom is insufficient)
#   - Linux fork-based DataLoader (default) — workers share the array via CoW,
#     physical pages are never duplicated regardless of num_workers
#
# Recommended for HPC nodes where RAM >> dataset size.
preload = True
check_ram = True   # set False to skip the pre-flight RAM check

# ---- Model / training hyperparams ----
device = "auto"
input_dim = 4096
final_dim = 512
recon_seq_len = 64
recon_hidden_dim = 256
recon_lambda = 1.0

max_epochs = 50
batch_size = 512
sub_batch_size = 64
lr = 1e-5
temperature = 0.25

num_workers = 8
persistent_workers = True

checkpoint_subdir = (
    "logprob_recon_contrastive_natural_questions"
    if use_logprob_features
    else "contrastive_natural_questions"
    )
checkpoint_dir = os.path.join("checkpoints", checkpoint_subdir)

def build_model_for_mode():
    if use_logprob_features:
        return LogprobReconProgressiveCompressor(
            input_dim=input_dim,
            final_dim=final_dim,
            input_dropout=0.3,
            recon_seq_len=recon_seq_len,
            recon_hidden_dim=recon_hidden_dim,
            recon_lambda=recon_lambda,
        )
    return ProgressiveCompressor(
        input_dim=input_dim,
        final_dim=final_dim,
        input_dropout=0.3,
    )

print(f"Inference JSON     : {inference_json}")
print(f"Eval JSON          : {eval_json}")
print(f"Raw eval JSONL     : {raw_eval_jsonl}")
print(f"Activations (zarr) : {activations_path}")
print(f"Training mode      : {training_mode}")
print(f"Preload            : {preload}  (check_ram={check_ram})")
print(f"Checkpoint dir     : {checkpoint_dir}")

_act_path = Path(activations_path)
print(f"Activations exists : {_act_path.exists()}")
if _act_path.exists():
    print(f"Activations abs path: {_act_path.resolve()}")
else:
    print("WARNING: activations_path does not exist in this runtime environment.")

In [ ]:
# ---- Build ActivationParser-compatible eval JSON for Natural Questions ----
def _load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def build_eval_for_activation_parser(
    inference_json_path: str,
    eval_json_path: str,
    raw_eval_jsonl_path: str,
    output_path: str,
):
    inference_path = Path(inference_json_path)
    if not inference_path.exists():
        raise FileNotFoundError(f"Missing inference file: {inference_path}")

    gendf = pd.read_json(inference_path, lines=True)
    n = len(gendf)
    if n == 0:
        raise RuntimeError("generation.jsonl is empty; nothing to train on")

    halu = None
    abstain = None
    source_used = None

    eval_path = Path(eval_json_path)
    if eval_path.exists():
        eval_payload = json.loads(eval_path.read_text(encoding="utf-8"))
        if (
            isinstance(eval_payload, dict)
            and "halu_test_res" in eval_payload
            and "abstantion" in eval_payload
            and len(eval_payload["halu_test_res"]) == n
            and len(eval_payload["abstantion"]) == n
        ):
            halu = [bool(x) for x in eval_payload["halu_test_res"]]
            abstain = [bool(x) for x in eval_payload["abstantion"]]
            source_used = "eval_results.json"

    if halu is None:
        raw_eval_path = Path(raw_eval_jsonl_path)
        if raw_eval_path.exists():
            raw_rows = _load_jsonl(raw_eval_path)
            if len(raw_rows) == n:
                halu = [
                    bool(r.get("is_hallucination", not bool(r.get("is_correct", False))))
                    for r in raw_rows
                ]
                abstain = [False] * n
                source_used = "raw_eval_res.jsonl"

    if halu is None:
        # Last-resort fallback: same string-match rule used by Natural Questions evaluation.
        halu = []
        for _, row in gendf.iterrows():
            answer = str(row.get("answer", "")).strip().lower()
            generation = str(row.get("generation", "")).strip().lower()
            is_correct = bool(answer) and (answer in generation)
            halu.append(not is_correct)
        abstain = [False] * n
        source_used = "generation.jsonl substring fallback"

    refusal_count = int(sum(abstain))
    hallu_count = int(sum(halu))
    correct_count = int(n - hallu_count)
    denom_not_abstain = max(1, n - refusal_count)

    compat_payload = {
        "evaluator_abstantion": "natural_questions_string_matching",
        "evaluator_hallucination": "natural_questions_string_matching",
        "abstantion": abstain,
        "halu_test_res": halu,
        "total_count": int(n),
        "accurate_count": correct_count,
        "hallu_count": hallu_count,
        "refusal_count": refusal_count,
        "correct_rate": float(correct_count / max(1, n)),
        "halu_rate_not_abstain": float(hallu_count / denom_not_abstain),
        "refusal_rate": float(refusal_count / max(1, n)),
    }

    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(compat_payload, indent=2), encoding="utf-8")

    print(f"Wrote parser-compatible eval file: {out_path}")
    print(f"  Source used: {source_used}")
    print(f"  Total samples: {n}")
    print(f"  Hallucinated: {hallu_count}")
    print(f"  Correct: {correct_count}")
    return str(out_path)


if not Path(activations_path).exists():
    raise FileNotFoundError(
        f"Missing activations store: {activations_path}\n"
        "Run inference first with activation logging enabled."
    )

eval_json_for_training = build_eval_for_activation_parser(
    inference_json_path=inference_json,
    eval_json_path=eval_json,
    raw_eval_jsonl_path=raw_eval_jsonl,
    output_path=eval_json_for_training,
    )

print(f"Eval JSON for training: {eval_json_for_training}")

## Load datasets

Set `training_mode` in Cell 3 to control behavior:

1. `regular`: standard supervised contrastive training
2. `logprob`: includes response logprob features for reconstruction loss

When using `logprob`, response logprobs must exist in the activation store.

In [ ]:
ap = ActivationParser(
    inference_json=inference_json,
    eval_json=eval_json_for_training,
    activations_path=activations_path,
    logger_type="zarr",
    verbose=False,
    )

include_response_logprobs = use_logprob_features

train_dataset = ap.get_dataset(
    "train",
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=num_views,
    backend=backend,
    include_response_logprobs=include_response_logprobs,
    response_logprobs_top_k=response_logprobs_top_k,
    preload=preload,
    check_ram=check_ram,
    )
test_dataset = ap.get_dataset(
    "test",
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=num_views,
    backend=backend,
    include_response_logprobs=include_response_logprobs,
    response_logprobs_top_k=response_logprobs_top_k,
    preload=preload,
    check_ram=check_ram,
    )

print(f"mode    : {training_mode}")
print(f"preload : {preload}  ({type(train_dataset).__name__})")
print(f"train   : {len(train_dataset)}")
print(f"test    : {len(test_dataset)}")
print(ap.df["halu"].value_counts(dropna=False).rename(index={0: "non-halu", 1: "halu"}))

In [ ]:
# ---- Validate optional logprob features ----
if not use_logprob_features:
    print("training_mode='regular': skipping logprob feature validation.")
else:
    if len(train_dataset) == 0:
        raise RuntimeError("Train dataset is empty; cannot validate logprobs.")

    # Quick per-sample probe on sample 0
    _probe = train_dataset[0]
    assert "response_token_logprobs" in _probe, (
        "response_token_logprobs not found in dataset items. "
        "Re-run inference with response logprobs enabled before using logprob mode."
        )

    _lp = _probe["response_token_logprobs"]
    _mask = _probe.get("response_logprob_mask")
    _valid_lp = _lp[~_lp.isnan()]
    _n_valid = int(_mask.sum()) if _mask is not None else int(_valid_lp.numel())

    print(f"Sample[0] logprob tensor shape : {_lp.shape}")
    print(f"Sample[0] valid tokens         : {_n_valid} / {_lp.shape[0]}")
    if _valid_lp.numel() > 0:
        print(f"Sample[0] logprob range        : [{_valid_lp.min():.3f}, {_valid_lp.max():.3f}]")
    else:
        print("Sample[0] logprob range        : [n/a, n/a] (all NaN)")

    # Coverage audit: determine if NaNs are isolated or widespread
    _scan_n = min(512, len(train_dataset))
    _valid_counts = []
    for _i in range(_scan_n):
        _item = train_dataset[_i]
        _vals = _item["response_token_logprobs"]
        _item_mask = _item.get("response_logprob_mask")
        _count = int(_item_mask.sum()) if _item_mask is not None else int((~_vals.isnan()).sum())
        _valid_counts.append(_count)

    _samples_with_valid = sum(c > 0 for c in _valid_counts)
    _samples_all_nan = _scan_n - _samples_with_valid
    _coverage_pct = 100.0 * _samples_with_valid / max(1, _scan_n)

    print("\nLogprob coverage audit")
    print(f"  Scanned samples              : {_scan_n}")
    print(f"  Samples with >=1 valid token : {_samples_with_valid}")
    print(f"  Samples all-NaN              : {_samples_all_nan}")
    print(f"  Coverage                     : {_coverage_pct:.2f}%")

    if _samples_with_valid == 0:
        raise RuntimeError(
            "All scanned samples have zero valid logprob tokens. "
            "This indicates a systemic issue (not just one sample). "
            "Common causes: wrong activations_path, logprob logging disabled during inference, or stale kernel state."
            )

    if _samples_all_nan > 0:
        print("Note: Some samples are all-NaN, but not all. This is a partial coverage issue.")
    else:
        print("All scanned samples have valid logprobs.")

    # Verify collator still exposes logprob key
    _probe_batch = _contrastive_collate_with_logprob([train_dataset[i] for i in range(min(4, len(train_dataset)))])
    assert "logprob" in _probe_batch, (
        "Collator did not produce a 'logprob' key. "
        "Check that _contrastive_collate_with_logprob sees response_token_logprobs."
        )
    print(f"Collated logprob shape: {_probe_batch['logprob'].shape}  (B, L)")

    del _probe, _lp, _mask, _valid_lp, _n_valid
    del _scan_n, _valid_counts, _samples_with_valid, _samples_all_nan, _coverage_pct
    del _probe_batch

In [ ]:
# ---- Build model ----
model = build_model_for_mode()

total_params = sum(p.numel() for p in model.parameters())
if use_logprob_features:
    encoder_params = sum(p.numel() for p in model.encoder.parameters())
    decoder_params = sum(p.numel() for p in model.decoder.parameters())
else:
    encoder_params = total_params
    decoder_params = 0

print(f"Total params  : {total_params:,}")
print(f"Encoder params: {encoder_params:,}")
if use_logprob_features:
    print(f"Decoder params: {decoder_params:,}  (discarded at inference)")
else:
    print("Decoder params: 0  (regular mode)")
model

In [ ]:
# ---- Train ----
train_device = device if device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")

if use_logprob_features:
    train_contrastive_logprob_recon(
        model,
        train_dataset,
        test_dataset=test_dataset,
        epochs=max_epochs,
        batch_size=batch_size,
        sub_batch_size=sub_batch_size,
        lr=lr,
        temperature=temperature,
        device=train_device,
        num_workers=num_workers,
        persistent_workers=persistent_workers,
        use_labels=True,
        ignore_label=outlier_class,
        checkpoint_dir=checkpoint_dir,
        snapshot_every=10,
        snapshot_keep_last=5,
        recon_lambda=recon_lambda,
    )
else:
    train_contrastive(
        model,
        train_dataset,
        test_dataset=test_dataset,
        epochs=max_epochs,
        batch_size=batch_size,
        sub_batch_size=sub_batch_size,
        lr=lr,
        temperature=temperature,
        device=train_device,
        num_workers=num_workers,
        persistent_workers=persistent_workers,
        use_labels=True,
        ignore_label=outlier_class,
        checkpoint_dir=checkpoint_dir,
        snapshot_every=10,
        snapshot_keep_last=5,
    )

## OOD evaluation

`forward()` on `LogprobReconProgressiveCompressor` is identical to `ProgressiveCompressor`
— the decoder is not called.  All existing evaluation utilities work unchanged.

In [ ]:
eval_device = device if device != 'auto' else ('cuda' if torch.cuda.is_available() else 'cpu')

# Eval datasets — slice target layers from the already-preloaded cache (no disk I/O).
train_dataset_for_inference = train_dataset.slice_layers(target_layers)
eval_dataset = test_dataset.slice_layers(target_layers)

train_loader_for_baseline = DataLoader(train_dataset_for_inference, batch_size=64, shuffle=False)
eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)

model_for_eval = model.to(eval_device)
model_for_eval.eval()

# num_workers=0 for eval: data is in-memory, forking workers adds no benefit
# and causes process cleanup noise from the training DataLoader's persistent workers.
eval_num_workers = 0

ood_eval = MultiMetricHallucinationEvaluator(
    activation_parser_df=ap.df,
    train_data_loader=train_loader_for_baseline,
    layers=None,
    batch_size=256,
    sub_batch_size=64,
    device=eval_device,
    num_workers=eval_num_workers,
    persistent_workers=False,
    outlier_class=outlier_class,
    metrics=[
        'cosine',
        'mds',
        {
            'metric': 'knn',
            'kwargs': {
                'k': 50,
                'metric': 'euclidean',
                'calibrate_k': True,
                'k_candidates': [50, 100, 200, 500, 1000],
                'max_train_size': 200000,
                'sample_seed': 0,
            },
            'train_selection': 'all',
        },
    ],
)
ood_stats = ood_eval.compute(eval_loader, model_for_eval)
print('OOD metrics:', ood_stats)

## Evaluate across epoch snapshots

Load each saved snapshot, run OOD evaluation, and compare metrics across training epochs.
Also tracks the train-time reconstruction loss from each checkpoint to diagnose
how well the auxiliary decoder steered the encoder.

In [ ]:
snapshot_pattern = os.path.join(checkpoint_dir, 'contrastive_snapshot_epoch_*.pt')
snapshot_files = sorted(glob.glob(snapshot_pattern))

last_ckpt = os.path.join(checkpoint_dir, 'contrastive_last.pt')
if os.path.exists(last_ckpt):
    snapshot_files.append(last_ckpt)
snapshot_files = sorted(set(snapshot_files))

def _parse_epoch(path):
    m = re.search(r'epoch_(\d+)', os.path.basename(path))
    return f'epoch_{int(m.group(1))}' if m else 'last'

snapshot_info = [(p, _parse_epoch(p)) for p in snapshot_files]
print(f'Found {len(snapshot_info)} checkpoint(s):')
for p, label in snapshot_info:
    print(f'  {label:>12s}  ->  {p}')

In [ ]:
if "train_loader_for_baseline" not in dir():
    train_dataset_for_inference = train_dataset.slice_layers(target_layers)
    train_loader_for_baseline = DataLoader(train_dataset_for_inference, batch_size=64, shuffle=False)
if "eval_loader" not in dir():
    eval_dataset = test_dataset.slice_layers(target_layers)
    eval_loader = DataLoader(eval_dataset, batch_size=64, shuffle=False)
if "eval_num_workers" not in dir():
    eval_num_workers = 0

all_results = []

for ckpt_path, epoch_label in tqdm(snapshot_info, desc="Evaluating snapshots"):
    ckpt = torch.load(ckpt_path, map_location=eval_device)

    snapshot_model = build_model_for_mode()
    snapshot_model.load_state_dict(ckpt["model_state_dict"])
    snapshot_model = snapshot_model.to(eval_device)
    snapshot_model.eval()

    ood_eval = MultiMetricHallucinationEvaluator(
        activation_parser_df=ap.df,
        train_data_loader=train_loader_for_baseline,
        layers=None,
        batch_size=256,
        sub_batch_size=64,
        device=eval_device,
        num_workers=eval_num_workers,
        persistent_workers=False,
        outlier_class=outlier_class,
        metrics=[
            "cosine",
            "mds",
            {
                "metric": "knn",
                "kwargs": {
                    "k": 50,
                    "metric": "euclidean",
                    "calibrate_k": True,
                    "k_candidates": [50, 100, 200, 500, 1000],
                    "max_train_size": 200000,
                    "sample_seed": 0,
                },
                "train_selection": "all",
            },
        ],
    )
    stats = ood_eval.compute(eval_loader, snapshot_model)

    epoch_num = int(re.search(r"\d+", epoch_label).group()) if re.search(r"\d+", epoch_label) else 9999
    row = {
        "checkpoint": epoch_label,
        "epoch": epoch_num,
        "path": ckpt_path,
        "train_loss": ckpt.get("train_loss"),
        "train_supcon": ckpt.get("train_supcon"),
        "train_recon": ckpt.get("train_recon"),
        "train_intra_cos": ckpt.get("train_intra_cos"),
        "train_intra_inter_margin": ckpt.get("train_intra_inter_margin"),
        "test_loss": ckpt.get("test_loss"),
        "recon_lambda": ckpt.get("recon_lambda", recon_lambda if use_logprob_features else None),
    }
    row.update(stats)
    all_results.append(row)

    supcon_val = ckpt.get("train_supcon")
    recon_val = ckpt.get("train_recon")
    if supcon_val is not None:
        recon_text = f"{float(recon_val):.4f}" if recon_val is not None else "n/a"
        logger.info(
            f"[{epoch_label}] supcon={float(supcon_val):.4f} recon={recon_text} | {stats}"
        )
    else:
        logger.info(
            f"[{epoch_label}] train_loss={ckpt.get('train_loss')} | {stats}"
        )

results_df = pd.DataFrame(all_results).sort_values("epoch").reset_index(drop=True)
print("\n=== Results across snapshots ===")
results_df

In [ ]:
import matplotlib.pyplot as plt

meta_cols = {'checkpoint', 'epoch', 'path', 'recon_lambda'}
metric_cols = [c for c in results_df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(results_df[c])]

# Separate training diagnostics from OOD metrics for cleaner layout
train_diag_cols = [c for c in metric_cols if c.startswith('train_')]
ood_cols = [c for c in metric_cols if c not in train_diag_cols]

def _plot_group(cols, title):
    if not cols:
        return
    fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), 4), squeeze=False)
    for ax, col in zip(axes.flatten(), cols):
        ax.plot(results_df['epoch'], results_df[col], marker='o', linewidth=1.5)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(col)
        ax.set_title(col)
        ax.grid(True, alpha=0.3)
    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

_plot_group(train_diag_cols, 'Training diagnostics vs Epoch')
_plot_group(ood_cols, 'OOD metrics vs Epoch')